# 03 - Where Markowitz Breaks

Replace the true moments with sample estimates and watch the optimizer go wild. This is the empirical face of the error-maximization phenomenon described by Michaud (1989).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from markowitz import MeanVariance

rng = np.random.default_rng(7)
n = 20

# Ground-truth moments.
mu_true = rng.uniform(0.04, 0.10, size=n)
A = rng.standard_normal((n, n))
Sigma_true = A @ A.T / n + 0.05 * np.eye(n)


## Draw finite samples and refit

We draw $T = 60$ monthly observations from the true distribution, form $\hat\mu, \hat\Sigma$, and solve the plug-in MVO. Then we repeat for many independent draws.

In [ ]:
T = 60
n_draws = 200
weights_history = np.empty((n_draws, n))

L = np.linalg.cholesky(Sigma_true)
for k in range(n_draws):
    R = mu_true + (rng.standard_normal((T, n)) @ L.T)
    mu_hat = R.mean(axis=0)
    Sigma_hat = np.cov(R, rowvar=False, ddof=1)
    res = MeanVariance(risk_aversion=3.0, long_only=False).fit(mu_hat, Sigma_hat)
    weights_history[k] = res.weights_


## Distribution of weights across draws

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(weights_history, showfliers=False)
ax.axhline(0.0, color='black', lw=0.5)
ax.set_xlabel('asset index')
ax.set_ylabel('weight across 200 sample draws')
ax.set_title(f'Plug-in MVO weights, T={T}, n={n}')
fig.tight_layout()


## Quantifying the instability

A useful diagnostic is the per-asset standard deviation of weights across draws. With well-estimated moments this should be small; here it is huge relative to the magnitude of any plausible portfolio.

In [ ]:
weight_std = weights_history.std(axis=0)
print('mean |w|:        ', np.abs(weights_history).mean())
print('mean weight std: ', weight_std.mean())
print('max  weight std: ', weight_std.max())


## What's going on?

Each new sample slightly perturbs $\hat\mu$ and $\hat\Sigma$, but the optimizer's response is enormous because $\hat\Sigma^{-1}$ amplifies directions with small eigenvalues. Notebook 04 fixes this by shrinking $\hat\Sigma$ toward a structured target.